# A1.2 · The controls, and where each one binds

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

Builds on **[A1.1 · What an agentic system is actually made of](https://spbreed.github.io/cyber-commons/lessons/A1.1.html)**.

| | |
|---|---|
| Open-source tooling | OPA, SPIFFE/SPIRE, Falco, agentgateway |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


You now have the parts. This lesson is the map: **seven controls, and the
component each one binds to.** Read it once here and the rest of Function A is
just detail.

| Control | Binds to | Answers | Taught in |
|---|---|---|---|
| **Identity** | the agent→tool edge | who is calling, and for whom | A2 |
| **Default-deny authorization** | the tool call | may *this* caller do *this* | A3.1 |
| **Sandboxed execution** | the loop's runtime | where model-written code runs | A3.2–A3.3 |
| **Tool and MCP trust** | tool descriptions and results | what a third party may say | A3.4–A3.5 |
| **Egress control** | the network edge | where data may go | A3.6 |
| **Containment** | the loop itself | how it stops | A3.7 |
| **Audit** | every edge | what you can reconstruct afterwards | D2, E1 |

Two things about this table are worth more than the table.

**Controls are not interchangeable.** A control placed in the wrong layer looks
like a control and holds nothing. A prompt that says "never email customer data"
is not egress control; it is a suggestion to a component that holds no authority
in the first place.

**Audit prevents nothing.** It is in the list because after an incident it is
the only thing that can answer *which user caused this* — and a system that
cannot answer that is not defensible even when nothing has gone wrong. Prevention
and reconstruction are different jobs, and confusing them is how teams end up
with neither.

Below, each control is removed one at a time to see which attacks stop being
stopped. The result is not the one most people expect.

## 2 · The controls, and the attacks they stop\n\nTen attacks that actually happen, and for each one the set of controls that would stop it. Any single control in the set is enough — which is what makes the counting interesting.

In [ ]:
CONTROLS = {
 "identity":    ("who is calling, and on whose behalf",        "A2"),
 "authz":       ("default-deny: may this caller do this",      "A3.1"),
 "sandbox":     ("where model-written code is allowed to run", "A3.2-A3.3"),
 "tool_trust":  ("what a tool description may cause",          "A3.4-A3.5"),
 "egress":      ("where data is allowed to go",                "A3.6"),
 "containment": ("how the loop is stopped",                    "A3.7"),
 "audit":       ("what can be reconstructed afterwards",       "D2, E1"),
}

# attack -> the controls that would stop it. Any one of them suffices.
ATTACKS = {
 "retrieved document tells the agent to email data out": {"egress", "authz"},
 "read-only user's agent deletes production rows":       {"identity", "authz"},
 "model-written code shells out to curl":                {"sandbox", "egress"},
 "MCP tool description carries an instruction":          {"tool_trust"},
 "every agent shares one service account":               {"identity"},
 "runaway loop burns the budget overnight":              {"containment"},
 "agent writes outside its workspace":                   {"sandbox"},
 "token replayed after the user logged out":             {"identity"},
 "third-party MCP server exfiltrates over its own socket":{"egress", "tool_trust"},
 "nobody can tell which user caused a deletion":         set(),
}

print(f"{'control':13s}{'taught in':12s}what it answers")
for name in sorted(CONTROLS):
    what, where = CONTROLS[name]
    print(f"{name:13s}{where:12s}{what}")

print(f"\n{len(ATTACKS)} attacks, and what stops each")
for attack in sorted(ATTACKS):
    stops = ATTACKS[attack]
    print(f"   {attack:56s}{', '.join(sorted(stops)) or 'NOTHING PREVENTS IT'}")

## 3 · Remove one control at a time\n\nAn attack is stopped while *any* of its controls is present. Take one control away and count what walks through.

In [ ]:
def unstopped(present):
    """Attacks nothing in `present` can stop."""
    return sorted(a for a, stops in ATTACKS.items() if not (stops & present))

ALL = set(CONTROLS)
baseline = unstopped(ALL)
print(f"with every control in place, unstopped: {len(baseline)}")
for a in baseline:
    print(f"   {a}")

print("\nremove one control:")
print(f"{'removed':13s}{'newly unstopped':>17s}  which")
rows = []
for c in sorted(CONTROLS):
    now = unstopped(ALL - {c})
    new = [a for a in now if a not in baseline]
    rows.append((c, new))
    print(f"{c:13s}{len(new):>17d}  {'; '.join(a[:44] for a in new) or '-'}")

assert all(a in unstopped(set()) for a in ATTACKS), "removing everything stops nothing"

## 4 · Read the result carefully\n\nThree things fall out of that table, and none of them is 'install more controls'.

In [ ]:
load_bearing = sorted(((len(new), c) for c, new in rows), reverse=True)
print("controls ranked by what they alone stop:")
for n, c in load_bearing:
    print(f"   {c:13s}{n}")

single = {a for a, s in ATTACKS.items() if len(s) == 1}
doubled = {a for a, s in ATTACKS.items() if len(s) >= 2}
print(f"\nattacks stopped by exactly one control : {len(single)}")
print(f"attacks stopped by two or more          : {len(doubled)}")
print(f"attacks no control prevents             : {len([a for a,s in ATTACKS.items() if not s])}")

print()
best = max(CONTROLS, key=lambda c: (len([a for a, s in ATTACKS.items() if c in s]), c))
covered = len([a for a, s in ATTACKS.items() if best in s])
print(f"1. No control stops everything. The best single one is {best}, and it")
print(f"   stops {covered} of {len(ATTACKS)} - the rest walk straight past it.")
print("2. The attacks stopped by two controls survive losing either one. That")
print("   is what defence in depth actually buys - not more stopping, but")
print("   stopping that tolerates one control being wrong.")
print("3. Removing `audit` newly unstops nothing at all, and the attack it")
print("   answers is unstopped either way. Audit does not prevent; it explains.")
print("   A system that cannot say which user caused a deletion is undefensible")
print("   even on a day when nothing went wrong.")
assert dict(rows)["audit"] == [], "audit prevents nothing - that is the point"
assert len(single) >= 4

## 5 · Where it breaks — the control in the wrong layer\n\nThe most common failure is not a missing control. It is a control placed where it has no authority.

In [ ]:
MISPLACED = {
 "a system prompt saying 'never email customer data'": "egress",
 "a tool description saying 'only for admins'":        "authz",
 "asking the model to refuse unsafe code":             "sandbox",
 "logging the tool call inside the agent's own store": "audit",
}
print(f"{'what a team ships':52s}{'believed to be':16s}holds?")
for shipped, believed in sorted(MISPLACED.items()):
    print(f"{shipped:52s}{believed:16s}NO")
print()
print("Each of these binds to the model or to the agent's own process - the")
print("components with no authority and no independence. The model cannot")
print("enforce egress because it never opens the socket; the agent cannot be")
print("its own audit log because it can write to it.")
print()
print("A control is only a control if the component it binds to is one the")
print("attacker does not already own by the time it matters.")

effective = {c: (c in CONTROLS) for c in MISPLACED.values()}
print(f"\nnamed correctly as categories: {sorted(effective)}")
print("placed correctly in these examples: none")
assert all(effective.values()), "every category here is real - the placement is not"

## 6 · Verify — the map is the reading order\n\nWhere each control is taught, in the order the rest of Function A takes them.

In [ ]:
ORDER = ["identity", "authz", "sandbox", "tool_trust", "egress",
         "containment", "audit"]
print(f"{'#':>2}  {'control':13s}{'taught in':12s}stops on its own")
for i, c in enumerate(ORDER, 1):
    alone = [a for a, s in ATTACKS.items() if s == {c}]
    print(f"{i:>2}  {c:13s}{CONTROLS[c][1]:12s}{len(alone)}")
assert ORDER[0] == "identity", "identity comes first: the rest are predicates on it"
print()
print("Identity is first because every control after it is a predicate that")
print("takes a caller as its argument. 'May this caller do this' is unanswerable")
print("while the answer to 'who is calling' is a shared service account.")
print()
print("Next: A2 answers that question properly - human identity, then workload")
print("identity, then why agents need their own.")

## What you just proved

Seven controls print with the component each binds to and the lesson that teaches it. Removing one control at a time shows identity alone stopping two attacks nothing else covers, four attacks resting on a single control, and `audit` newly unstopping nothing — because audit does not prevent, it explains. Four commonly shipped 'controls' are shown binding to components that hold no authority.

## Your turn

Take the seven rows and mark, for one system you run, which control is actually enforced and by which component. The rows you cannot name a component for are the ones a review will find.

---

**Next → [A1.3 · Architecture review when the system acts](https://spbreed.github.io/cyber-commons/lessons/A1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*